**Purpose:** Run the final news model over the corpus for feature extraction.

**Outputs:** `sentiment_news_{str(BUCKET_TO_DO)}_{str(current_hash)}.parquet`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.config import PROJECT_ROOT


https://www.kaggle.com/code/hugoverssimo/news-fullclassify

and taking in total (for all timestamp of news) 108 hours 

In [ ]:
!pip install torch==2.2.2+cu118 torchvision==0.17.2+cu118 torchaudio==2.2.2+cu118 --index-url https://download.pytorch.org/whl/cu118
!pip install transformers==4.36.2 scikit-learn pandas numpy
!pip install --upgrade --no-deps requests accelerate huggingface_hub==0.17.4 spacy==3.8.7 pyarrow
!python -m spacy download en_core_web_sm

import torch

import warnings
warnings.filterwarnings("ignore")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected")

In [ ]:
LAST_HASH = 1497700634026591235132812425252607120
BUCKET_TO_DO = 1

In [ ]:
import os, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AdamW
from tqdm import tqdm
import time
import re
import hashlib
import requests, zipfile, io
import spacy

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

label2id = {
    'negative': 0,
    'neutral': 1,
    'positive': 2
}

feature2id = {
    'Communication Services': 0,
    'Consumer Discretionary': 1,
    'Consumer Staples': 2,
    'Energy': 3,
    'Financials': 4,
    'Health Care': 5,
    'Industrials': 6,
    'Information Technology': 7,
    'Materials': 8,
    'Real Estate': 9,
    'Utilities': 10
}

# ================= MODEL =================
class FinBERTWithFeature(nn.Module):
    def __init__(self, num_labels, num_features, feature_dim=32, dropout=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained("ProsusAI/finbert")

        for name, param in self.bert.named_parameters():
            if "encoder.layer.10" not in name and "encoder.layer.11" not in name:
                param.requires_grad = False

        self.feature_emb = nn.Embedding(num_features, feature_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size + feature_dim, num_labels)

    def forward(self, input_ids, attention_mask, feature_id):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0]
        feat = self.feature_emb(feature_id)
        x = torch.cat([cls, feat], dim=1)
        x = self.dropout(x)
        return self.classifier(x)


tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

model = FinBERTWithFeature(
    len(label2id),
    11, # NUM_SECTORS
    feature_dim=32,
    dropout=0.5
).to(DEVICE)

state_dict = torch.load(
    str(PROJECT_ROOT / "02_sentiment/news/vf2_sector_embeddings/final_model.pt"),
    map_location=DEVICE
)

model.load_state_dict(state_dict)
model.eval()

def predict_sentences(
    sentences,
    sector,
    model,
    tokenizer,
    label2id,
    feature2id,
    max_len=128,
    device="cpu"
):
    
    if not sentences:
        return []

    model.eval()

    # Encode text batch
    enc = tokenizer(
        sentences,
        truncation=True,
        padding=True,
        max_length=max_len,
        return_tensors="pt"
    )

    # Same sector for all sentences
    feature_id = torch.tensor(
        [feature2id[sector]] * len(sentences),
        dtype=torch.long
    )

    enc = {k: v.to(device) for k, v in enc.items()}
    feature_id = feature_id.to(device)

    with torch.no_grad():
        logits = model(
            enc["input_ids"],
            enc["attention_mask"],
            feature_id
        )
        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

    id2label = {v: k for k, v in label2id.items()}

    return [
        {
            "sentence": s,
            "prediction": id2label[p.item()],
            "confidence": probs[i, p].item()
        }
        for i, (s, p) in enumerate(zip(sentences, preds))
    ]

In [ ]:
nlp = spacy.load("en_core_web_sm")

def deterministic_shuffle(lista_datas):
    def digital_root(n: int) -> int:
        while n > 9:
            s = 0
            while n > 0:
                s += n % 10
                n //= 10
            n = s
        return n
    dates = lista_datas
    encoded = {
        int(hashlib.md5(x.encode()).hexdigest(), 16): x
        for x in dates
    }    
    encoded_sorted = dict(sorted(encoded.items()))
    encoded_split = {k: (digital_root(k), v) for k,v in encoded_sorted.items()}
    return encoded_split

def clean_quotations(text):
    text = re.sub(r'\d+\|\d+\|\|', '', text)
    text = re.sub(r'#\d+', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s([.,!?;:])', r'\1', text)
    return text.strip()
    
def lemmatize_quotation(text):
    doc = nlp(text)
    lemmas = [token.lemma_ for token in doc]
    return " ".join(lemmas)

def load_gdelt_news(timestamp, sources, sector_patterns, news_settings):
    url = f"http://data.gdeltproject.org/gdeltv2/{timestamp}.gkg.csv.zip"
    r = requests.get(url)
    try:
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            with z.open(z.namelist()[0]) as f:
                df = pd.read_csv(f, sep='\t', header=None, dtype=str, encoding='ISO-8859-1')
                df.columns = [
                    "GKGRECORDID", "DATE", "SourceCollectionIdentifier", "SourceCommonName",
                    "DocumentIdentifier", "Counts", "V2Counts", "Themes", "V2Themes",
                    "Locations", "V2Locations", "Persons", "V2Persons", "Organizations",
                    "V2Organizations", "V2Tone", "Dates", "GCAM", "SharingImage",
                    "RelatedImages", "SocialImageEmbeds", "SocialVideoEmbeds", "Quotations",
                    "AllNames", "Amounts", "TranslationInfo", "Extras"
                ]
                df = df[["GKGRECORDID", "DATE", "SourceCommonName", "Quotations"]]
                df = df[df["SourceCommonName"].isin(sources)]
                df = df[df["Quotations"].notna()]
                df["Quotations"] = df["Quotations"].map(clean_quotations)
                df = df.drop_duplicates(subset=["SourceCommonName", "Quotations"])
                df["search_Quotations"] = df["Quotations"].map(lemmatize_quotation)
                sector_dfs = {}
                for sector, pattern in sector_patterns.items():
                    sector_dfs[sector] = df[df["SourceCommonName"].isin(news_settings["SourceCommonName_filter"][sector])]
                    sector_dfs[sector] = sector_dfs[sector][sector_dfs[sector]["search_Quotations"].str.contains(pattern, na=False, flags=re.IGNORECASE)]
    except Exception as e:
        error_message = f"Failed to process {url}: {e}\n"
        with open("errors.log", "a") as log_file:
            log_file.write(error_message)
        return False, error_message
    return True, sector_dfs

with open(str(PROJECT_ROOT / "01_data/processed/news_0.json"), "r") as f:
    news_settings = json.load(f)

sector_patterns = {
    sector: r"(?<!\w)(?:" + "|".join(map(re.escape, words)) + r")(?!\w)"
    for sector, words in news_settings["Quotations_filter"].items()
}

sources = {source for lst in news_settings["SourceCommonName_filter"].values() for source in lst}

timestamps = deterministic_shuffle(news_settings["DATE_filter"])

max_duration = 10 * 3600
start_time = time.time()

timestamps_count = 0
records_count = 0
records = []

for current_hash, (bucket, timestamp) in timestamps.items():

    elapsed_time = time.time() - start_time
    if elapsed_time > max_duration:
        print(f"⏰ Time limit reached ({max_duration / 3600} hours). Stopping the loop.")
        break

    if LAST_HASH > current_hash or bucket != BUCKET_TO_DO:
        continue

    data = {
        "timestamp": timestamp
    }
        
    valid_news_dfs, news_dfs = load_gdelt_news(timestamp, sources, sector_patterns, news_settings)
    if not valid_news_dfs:
        data["error"] = news_dfs
        continue
    
    print(f"Running {timestamp} ({current_hash})")
    timestamps_count += 1
    
    for sector, df in news_dfs.items():
        data[sector] = {}

        try:
            quotes = df["Quotations"].tolist()

            labels = predict_sentences(
                quotes,
                sector=sector,
                model=model,
                tokenizer=tokenizer,
                label2id=label2id,
                feature2id=feature2id,
                device=DEVICE
            )

            results = {}
            for r in labels:
                if r["prediction"] not in results:
                    results[r["prediction"]] = []
                results[r["prediction"]].append(round(r["confidence"], 3))

            data[sector] = results
            records_count += len(quotes)

        except RuntimeError as e:
            with open("failed.txt", "w") as f:
                f.write(str(timestamp) + " : " + str(sector) + " : " + str(e) + "\n")
            data["error"] = f"Failed to process {timestamp} - {sector}: {e}"
            continue

    records.append({
            "timestamp": timestamp,
            "data": data
        })

df = pd.DataFrame(records)
df["data"] = df["data"].apply(json.dumps)
df.to_parquet(f"sentiment_news_{str(BUCKET_TO_DO)}_{str(current_hash)}.parquet", engine="pyarrow", index=False)

with open("last_hash.txt", "w") as f:
    f.write(str(current_hash))
    f.write("\n Timestamps count: " + str(timestamps_count))
    f.write("\n Records count: " + str(records_count))
print("END")